#Vector Space Model
### Vector Space Model (VSM)

**Vector Space Model (VSM)** adalah sebuah model representasi untuk dokumen yang sering digunakan dalam pemrosesan informasi dan pencarian informasi. Dalam model ini, dokumen dan istilah diwakili sebagai vektor dalam ruang berdimensi tinggi. Berikut adalah penjelasan lebih detail mengenai VSM:

#### 1. **Konsep Dasar**
- **Dokumen dan Kata sebagai Vektor**: Dalam VSM, setiap dokumen diwakili sebagai vektor, di mana setiap dimensi vektor sesuai dengan kata tertentu dalam korpus (kumpulan dokumen).
- **Frekuensi Kata**: Nilai setiap dimensi dalam vektor dokumen biasanya merupakan frekuensi kata (count), tetapi bisa juga menggunakan nilai yang lebih kompleks seperti **TF-IDF** (Term Frequency-Inverse Document Frequency) untuk memberi bobot pada kata berdasarkan pentingnya kata tersebut dalam konteks dokumen dan korpus.

#### 2. **Representasi**
- **Ruang Vektor**: Dengan VSM, semua dokumen dapat dipetakan ke dalam ruang vektor n-dimensi, di mana n adalah jumlah kata unik dalam korpus.
- **Contoh**: Jika ada tiga kata unik: "kucing", "anjing", dan "burung", dokumen yang berisi "kucing" dan "anjing" akan direpresentasikan sebagai vektor [1, 1, 0] (frekuensi masing-masing kata).

#### 3. **Penghitungan Similarity**
- **Cosine Similarity**: Salah satu cara untuk mengukur kesamaan antara dua vektor dokumen adalah dengan menggunakan cosine similarity, yang mengukur sudut antara dua vektor. Formula untuk cosine similarity adalah:

\[
\text{Cosine Similarity} = \frac{A \cdot B}{||A|| \cdot ||B||}
\]

  di mana \(A\) dan \(B\) adalah vektor dokumen, \(A \cdot B\) adalah hasil kali titik (dot product) antara kedua vektor, dan \(||A||\) dan \(||B||\) adalah norma (magnitudo) dari vektor \(A\) dan \(B\).

#### 4. **Keuntungan VSM**
- **Sederhana dan Mudah Dipahami**: Konsep representasi dokumen sebagai vektor intuitif dan mudah untuk diimplementasikan.
- **Mendukung Pencarian**: Memudahkan pencarian dan pengambilan informasi dengan menghitung kesamaan antara dokumen dan kueri.
- **Dapat Mengakomodasi Berbagai Dimensi**: Dapat diperluas untuk mencakup berbagai fitur dari dokumen, seperti metadata, tag, atau fitur lain yang relevan.

#### 5. **Keterbatasan VSM**
- **Kehilangan Makna**: VSM tidak mempertimbangkan konteks atau makna dari kata-kata. Misalnya, kata "bank" dapat merujuk pada lembaga keuangan atau tepi sungai, tetapi VSM tidak dapat membedakannya.
- **Dimensi Tinggi**: VSM dapat menghasilkan ruang vektor yang sangat besar, yang dapat menyebabkan masalah dalam komputasi dan penyimpanan (dikenal sebagai curse of dimensionality).
- **Ketidakpastian**: VSM tidak menangani ambiguitas atau sinonim dengan baik. Kata-kata yang berbeda tetapi memiliki arti sama akan dianggap sebagai fitur terpisah.

#### 6. **Aplikasi VSM**
- **Pencarian Informasi**: Digunakan dalam sistem pencarian untuk menemukan dokumen yang relevan berdasarkan kueri pengguna.
- **Klasifikasi Teks**: Digunakan dalam algoritma klasifikasi untuk mengategorikan dokumen ke dalam kelas yang berbeda.
- **Rekomendasi Konten**: Digunakan dalam sistem rekomendasi untuk memberikan saran konten yang relevan kepada pengguna.

### Kesimpulan
Vector Space Model adalah salah satu pendekatan yang paling umum dan efektif untuk representasi teks dalam pemrosesan informasi. Meskipun memiliki beberapa keterbatasan, kekuatan dan kesederhanaannya menjadikannya alat yang sangat berguna dalam berbagai aplikasi di bidang pemrosesan bahasa alami dan pencarian informasi.

In [1]:
# Library untuk data manipulation
!pip install Sastrawi
import pandas as pd
from tqdm import tqdm
import re
import string

# Library untuk text preprocessing
import nltk
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt_tab')

# Library untuk text vectorization/TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

# Library untuk save model
import pickle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 3.2 MB/s eta 0:00:00


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


- sastrawi : stemming teks dalam bahasa indonesia.
- pandas : manipulasi dan analisis data dalam bentuk tabel.
- tqdm : melacak kemajuan ketika memproses data dalam jumlah besar.
- re : modul string
- stemming : proses mengubah kata menjadi bentuk dasarnya.
- **`from sklearn.feature_extraction.text import TfidfVectorizer`**: Mengimpor **TfidfVectorizer** dari pustaka **scikit-learn**. Ini digunakan untuk mengubah kumpulan dokumen teks menjadi matriks TF-IDF, yang merupakan representasi numerik dari teks berdasarkan frekuensi kata.

- **`import pickle`**: Mengimpor modul **pickle**, yang digunakan untuk serialisasi objek Python. Ini berguna untuk menyimpan model atau objek lain ke dalam file untuk digunakan nanti, tanpa harus melatih ulang model tersebut.

In [2]:
data = pd.read_csv("hasil_crowling.csv")
data.columns = data.columns.str.strip()
data

,Judul,Tanggal,Ringkasan,Kategori
0,"Viral Sopir Truk Nekat Lawan Arah di Jakut, Po...",16 Okt 2024 18:04,Dalam rekaman yang diunggah terlihat sopir tru...,Peristiwa
1,Polisi Pastikan 2 Tersangka Pencabulan di Pant...,16 Okt 2024 18:02,"Kabid Humas Polda Metro Jaya, Kombes Pol Ade A...",Peristiwa
2,Sejumlah Tokoh dan Pakar Siap Sukseskan Pelant...,16 Okt 2024 16:05,Mulusnya transisi pemerintahan dapat menjaga s...,Politik
3,"Kasus Emas Antam, Saksi Sebut Budi Said Lakuka...",16 Okt 2024 16:01,"Mantan Manajer Retail PT Antam, Nuning Septi W...",Peristiwa
4,5 Fakta Terkait Budi Gunawan Diberhentikan dar...,16 Okt 2024 16:00,Dewan Perwakilan Rakyat (DPR) RI telah menerim...,Peristiwa
5,"Pramono: Semua Ketum Parpol Teman Saya, Nanti ...",16 Okt 2024 15:51,"Calon Gubernur Jakarta, Pramono Anung mengaku ...",News
6,DPR Minta Herindra Bisa Netral saat Sudah Resm...,16 Okt 2024 15:41,"Kepada Herindra, Puan menitipkan pesan khusus ...",Peristiwa
7,"Pengamat Nilai Jika SK-ADT Pimpin Sulut, Dukun...",16 Okt 2024 15:25,Pengamat politik dan pemerintahan Sulawesi Uta...,Politik
8,"Cegah Banjir Terulang, Pramono Bakal Bebaskan ...",16 Okt 2024 15:22,Untuk mengatasi banjir di daerah bantaran kali...,News
9,Puan Maharani Masih Irit Bicara soal Pertemuan...,16 Okt 2024 15:03,"Ketua DPP PDI Perjuangan, Puan Maharani belum ...",Politik


- **`pd.read_csv("hasil_crowling.csv")`**: Menggunakan fungsi `read_csv` dari pustaka **Pandas** untuk membaca file CSV bernama `hasil_crowling.csv`. Data yang dibaca akan disimpan dalam variabel `data` dalam bentuk **DataFrame**.
- DataFrame adalah struktur data dua dimensi (mirip dengan tabel) yang terdiri dari baris dan kolom, sehingga mudah untuk dianalisis dan dimanipulasi.
- **`data.columns`**: Mengakses nama-nama kolom dari DataFrame `data`.
- **`data.columns.str.strip()`**: Menggunakan metode `str.strip()` untuk menghapus spasi di awal dan akhir nama kolom. Ini penting untuk memastikan bahwa nama kolom bersih dari karakter spasi yang tidak perlu, sehingga menghindari potensi masalah ketika merujuk atau memanipulasi kolom tersebut dalam analisis lebih lanjut.
- Baris ini akan menampilkan isi dari DataFrame `data` di output. Jika Anda menjalankan kode ini dalam lingkungan interaktif (seperti Jupyter Notebook), DataFrame akan ditampilkan dalam format tabel, menunjukkan semua baris dan kolom yang ada.


In [3]:
data = data.sample(frac = 1, ignore_index=True)

- **`data.sample()`**: Fungsi `sample()` digunakan untuk mengambil sampel dari DataFrame. Dalam hal ini, kita akan mengambil semua baris, tetapi urutannya akan diacak.
- **`frac=1`**: Parameter `frac` menentukan proporsi dari DataFrame yang ingin diambil. Dengan mengatur `frac=1`, kita memberitahu fungsi untuk mengambil 100% dari DataFrame, sehingga semua baris akan diambil.
- **`ignore_index=True`**: Parameter ini digunakan untuk mengabaikan indeks asli dari DataFrame dan memberikan indeks baru yang berurutan mulai dari 0. Ini berarti setelah pengacakan, DataFrame yang dihasilkan akan memiliki indeks yang dimulai dari 0 hingga n-1, di mana n adalah jumlah baris dalam DataFrame.

In [4]:
def clean_text(text):
	text = re.sub(r'((www\.[^\s]+)|(https?://[^\s]+))', ' ', text) # Menghapus https* and www*
	text = re.sub(r'@[^\s]+', ' ', text) # Menghapus username
	text = re.sub(r'[\s]+', ' ', text) # Menghapus tambahan spasi
	text = re.sub(r'#([^\s]+)', ' ', text) # Menghapus hashtags
	text = re.sub(r'rt', ' ', text) # Menghapus retweet
	text = text.translate(str.maketrans("","",string.punctuation)) # Menghapus tanda baca
	text = re.sub(r'\d', ' ', text) # Menghapus angka
	text = text.lower()
	text = text.encode('ascii','ignore').decode('utf-8') #Menghapus ASCII dan unicode
	text = re.sub(r'[^\x00-\x7f]',r'', text)
	text = text.replace('\n','') #Menghapus baris baru
	text = text.strip()
	return text

Fungsi `clean_text` di atas bertujuan untuk membersihkan teks dari berbagai elemen yang tidak diinginkan agar lebih siap untuk analisis teks atau pemrosesan lebih lanjut. Berikut adalah penjelasan dari setiap bagian kode tersebut:

### Penjelasan Kode

1. **Menghapus URL**
   ```python
   text = re.sub(r'((www\.[^\s]+)|(https?://[^\s]+))', ' ', text)
   ```
   - Menggunakan **regular expression (regex)** untuk mencari dan menghapus semua URL dari teks, baik yang dimulai dengan "www." maupun yang dimulai dengan "http://" atau "https://".

2. **Menghapus Username**
   ```python
   text = re.sub(r'@[^\s]+', ' ', text)
   ```
   - Menghapus semua mention atau username yang dimulai dengan `@`, diikuti oleh karakter non-spasi.

3. **Menghapus Spasi Berlebih**
   ```python
   text = re.sub(r'[\s]+', ' ', text)
   ```
   - Mengganti beberapa spasi berturut-turut dengan satu spasi, sehingga tidak ada spasi berlebih di dalam teks.

4. **Menghapus Hashtags**
   ```python
   text = re.sub(r'#([^\s]+)', ' ', text)
   ```
   - Menghapus semua hashtag yang diawali dengan `#`, diikuti oleh karakter non-spasi.

5. **Menghapus Kata 'rt'**
   ```python
   text = re.sub(r'rt', ' ', text)
   ```
   - Menghapus kata "rt" yang biasanya digunakan dalam konteks retweet pada media sosial.

6. **Menghapus Tanda Baca**
   ```python
   text = text.translate(str.maketrans("", "", string.punctuation))
   ```
   - Menghapus semua tanda baca menggunakan `str.maketrans()` dan `translate()`. Fungsi ini digunakan untuk membuat tabel transposisi yang akan menghapus semua karakter tanda baca.

7. **Menghapus Angka**
   ```python
   text = re.sub(r'\d', ' ', text)
   ```
   - Menghapus semua angka dari teks.

8. **Mengubah Teks Menjadi Huruf Kecil**
   ```python
   text = text.lower()
   ```
   - Mengubah semua karakter dalam teks menjadi huruf kecil untuk konsistensi.

9. **Menghapus Karakter ASCII dan Unicode**
   ```python
   text = text.encode('ascii','ignore').decode('utf-8')
   text = re.sub(r'[^\x00-\x7f]', r'', text)
   ```
   - Langkah pertama menghapus karakter yang tidak dapat direpresentasikan dalam ASCII, dan langkah kedua lebih lanjut menghapus karakter non-ASCII dari teks.

10. **Menghapus Baris Baru**
    ```python
    text = text.replace('\n', '')
    ```
    - Menghapus karakter baris baru dari teks.

11. **Menghapus Spasi di Awal dan Akhir**
    ```python
    text = text.strip()
    ```
    - Menghapus spasi yang tidak diinginkan di awal dan akhir teks.

12. **Mengembalikan Teks yang Sudah Dibersihkan**
    ```python
    return text
    ```

In [5]:
def stemming_indo(text):
	factory = StemmerFactory()
	stemmer = factory.create_stemmer()
	text = ' '.join(stemmer.stem(word) for word in text)
	return text


### Penjelasan Kode

1. **Membuat Objek Stemmer**
   ```python
   factory = StemmerFactory()
   stemmer = factory.create_stemmer()
   ```
   - `StemmerFactory` adalah kelas dari pustaka **Sastrawi**, yang digunakan untuk membuat objek stemmer.
   - Dengan memanggil `create_stemmer()`, Anda mendapatkan objek stemmer yang dapat digunakan untuk melakukan stemming pada kata-kata.

2. **Melakukan Stemming pada Setiap Kata**
   ```python
   text = ' '.join(stemmer.stem(word) for word in text)
   ```
   - Kode ini menggunakan list comprehension untuk menerapkan fungsi `stem` dari objek `stemmer` pada setiap kata dalam `text`.
   - `text` diharapkan adalah string yang berisi kalimat atau teks. Namun, fungsi ini belum melakukan tokenisasi, sehingga sebaiknya `text` sudah berupa daftar kata (misalnya, hasil dari tokenisasi).
   - `stemmer.stem(word)` akan mengembalikan bentuk dasar (stem) dari `word`.
   - `join()` kemudian menggabungkan kembali kata-kata yang telah distem menjadi satu string dengan spasi sebagai pemisah.

3. **Mengembalikan Teks yang Telah Distem**
   ```python
   return text
   ```
   - Fungsi mengembalikan teks yang telah mengalami proses stemming.

In [6]:
def clean_stopword(tokens):
	listStopword =  set(stopwords.words('indonesian'))
	removed = []
	for t in tokens:
		if t not in listStopword:
			removed.append(t)
	return removed


### Penjelasan Kode

1. **Mengambil Daftar Stopwords**
   ```python
   listStopword = set(stopwords.words('indonesian'))
   ```
   - `stopwords.words('indonesian')` menggunakan pustaka **nltk** untuk mendapatkan daftar stopwords dalam bahasa Indonesia. Stopwords adalah kata-kata umum (seperti "dan", "di", "yang") yang sering diabaikan dalam analisis teks karena tidak memberikan makna penting.
   - Dengan membungkusnya dalam `set()`, kita mengkonversi daftar tersebut menjadi set, yang mempercepat proses pencarian kata-kata ketika memeriksa apakah sebuah token merupakan stopword atau tidak.

2. **Inisialisasi Daftar untuk Menyimpan Token yang Diterima**
   ```python
   removed = []
   ```
   - Variabel `removed` diinisialisasi sebagai daftar kosong yang akan digunakan untuk menyimpan token yang bukan stopword.

3. **Menghapus Stopwords dari Daftar Token**
   ```python
   for t in tokens:
       if t not in listStopword:
           removed.append(t)
   ```
   - Loop ini iterasi melalui setiap token (`t`) dalam daftar `tokens`.
   - Jika token tersebut tidak ada dalam `listStopword`, maka token tersebut ditambahkan ke dalam daftar `removed`.

4. **Mengembalikan Daftar Token yang Diterima**
   ```python
   return removed
   ```
   - Fungsi mengembalikan daftar `removed`, yang berisi semua token yang bukan stopword.

### Contoh Penggunaan
Berikut adalah contoh penggunaan fungsi `clean_stopword` setelah tokenisasi teks:

```python
from nltk.tokenize import word_tokenize

In [7]:
import nltk
nltk.download('punkt')
def preprocess_text(content):
	result = []
	for text in tqdm(content):
		cleaned_text = clean_text(text)
		tokens = nltk.tokenize.word_tokenize(cleaned_text)
		cleaned_stopword = clean_stopword(tokens)
		stemmed_text = stemming_indo(cleaned_stopword)
		result.append(stemmed_text)
	return result

data['cleaned_text'] = preprocess_text(data['Ringkasan'])

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
100%|██████████| 53/53 [00:45<00:00,  1.16it/s]


1. **Impor Library**: Mengimpor library seperti NLTK untuk tokenisasi dan Sastrawi untuk stemming. Juga mengunduh model tokenizer.

2. **Fungsi `preprocess_text`**:
   - **Input**: Menerima daftar teks (kolom `Ringkasan`).
   - **Proses**:
     - **Pembersihan**: Menggunakan `clean_text` untuk menghapus URL, tanda baca, dan karakter tidak diinginkan.
     - **Tokenisasi**: Memecah teks menjadi kata-kata dengan `word_tokenize`.
     - **Penghapusan Stopwords**: Menghapus kata umum yang tidak berarti dengan `clean_stopword`.
     - **Stemming**: Mengubah kata menjadi bentuk dasarnya menggunakan `stemming_indo`.

3. **Output**: Mengembalikan daftar teks yang telah diproses dan menyimpannya di kolom baru `cleaned_text` pada DataFrame.

### Tujuan
Mempersiapkan teks untuk analisis lebih lanjut dengan membersihkan, tokenisasi, penghapusan kata henti, dan stemming, sehingga data lebih terstruktur dan informatif.

In [8]:
#mengambil 30 baris pertama dari dataframe dan menyimpannya ke dalam variabel data_train.
data_train = data[:30]

#Mengambil baris dari indeks 30 hingga 52 (total 23 baris) dan menyimpannya ke dalam variabel data_test. Ini berfungsi sebagai data pengujian (testing data).
data_test = data[30:53]
data_train

#Data Pelatihan (training data) digunakan untuk melatih model,
#Data Pengujian (testing data) digunakan untuk menguji kinerja model setelah pelatihan.

,Judul,Tanggal,Ringkasan,Kategori,cleaned_text
0,7 Fakta Terkini Terkait Kasus Pelecehan Seksua...,16 Okt 2024 14:00,Polres Metro Tangerang mengungkap kronologi te...,Peristiwa,polres metro tangerang ungkap kronologi bongka...
1,Puan Maharani Masih Irit Bicara soal Pertemuan...,16 Okt 2024 15:03,"Ketua DPP PDI Perjuangan, Puan Maharani belum ...",Politik,ketua dpp pdi juang puan maharani lugas konfir...
2,WSBP Dapat Kontrak Proyek Konstruksi Pembangun...,16 Okt 2024 07:08,Proyek Pembangunan UNIPI PERSIS ini nantinya d...,Peristiwa,proyek bangun unipi persis harap tingkat saran...
3,Jokowi Resmikan Bendungan Lausimeme Senilai Rp...,16 Okt 2024 10:17,Jokowi menegaskan pentingnya proyek pembanguna...,Peristiwa,jokowi proyek bangun habis anggar rp triliun
4,Herindra Bakal Dilantik Sebagai Kepala BIN Bar...,16 Okt 2024 12:24,"Sebelum tes dimulai pukul 11.00 WIB, Herindra ...",Peristiwa,tes wib herindra sapa awak media lambai tangan...
5,"Hari Parlemen Indonesia, Novita Hardini Soroti...",16 Okt 2024 14:31,Novita menegaskan bahwa kehadiran perempuan di...,Politik,novita hadir perempuan parlemen kaya isi kursi...
6,Program Asta Cita Prabowo-Gibran Diharap Bisa ...,16 Okt 2024 13:00,PB HMI berharap Prabowo dan Gibran tidak hanya...,Peristiwa,pb hmi harap prabowo gibran fokus bangun fisik...
7,"Bejat, Ayah Tiri di Cipondoh Tangerang Cabuli ...",16 Okt 2024 06:00,Pelaku berinisial MA (42) telah ditetapkan seb...,Megapolitan,laku inisial ma tetap sangka tahan mapolres me...
8,Profil Muhammad Herindra yang Ditunjuk Jokowi ...,16 Okt 2024 09:31,Herindra adalah lulusan Akademi Militer (Akmil...,Peristiwa,herindra lulus akademi militer akmil baik raih...
9,"Perjalanan Hidup Denny Tuejeh, dari Anak Sopir...",16 Okt 2024 05:45,Kisah Denny Tuejeh di panggung Pilkada Sulut 2...,News,kisah denny tuejeh panggung pilkada sulut jala...


In [9]:
def tfidf_vsm(data, kategori):
	tfidf = TfidfVectorizer()
	tfidf_matrix = tfidf.fit_transform(data)
	feature_names = tfidf.get_feature_names_out()

	df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
	df_tfidf.insert(0, 'Kategori Berita', kategori.reset_index(drop=True))

	return tfidf, df_tfidf

tfidf_model, df_tfidf = tfidf_vsm(data_train['cleaned_text'], data_train['Kategori'])

### Penjelasan Kode

1. **Fungsi `tfidf_vsm`**:
   - **Parameter**:
     - `data`: Kumpulan teks yang telah diproses (misalnya, ringkasan berita).
     - `kategori`: Kategori berita yang terkait dengan teks.
   - **Proses**:
     - `TfidfVectorizer()`: Membuat objek untuk menghitung TF-IDF dari data teks.
     - `fit_transform(data)`: Menghitung TF-IDF dari `data` dan mengubahnya menjadi matriks.
     - `get_feature_names_out()`: Mengambil nama fitur (kata) dari matriks TF-IDF.
     - `pd.DataFrame(...)`: Mengonversi matriks TF-IDF menjadi DataFrame, dengan kolom fitur sebagai nama kata.
     - `insert(...)`: Menambahkan kolom baru yang berisi kategori berita ke DataFrame TF-IDF. Kolom ini disisipkan di awal.

2. **Menggunakan Fungsi**:
   - `tfidf_model, df_tfidf = tfidf_vsm(data_train['cleaned_text'], data_train['Kategori'])`:
     - Memanggil fungsi `tfidf_vsm` dengan data yang sudah dibersihkan dan kategori dari `data_train`.
     - Hasilnya adalah model TF-IDF (`tfidf_model`) dan DataFrame TF-IDF (`df_tfidf`), yang berisi skor TF-IDF untuk setiap kata di setiap dokumen, beserta kategori berita yang bersangkutan.

### Tujuan
Tujuan dari kode ini adalah untuk:
- Mengubah teks yang sudah diproses menjadi bentuk numerik (matriks TF-IDF) yang dapat digunakan dalam model machine learning.
- Mengidentifikasi pentingnya setiap kata dalam konteks setiap dokumen (berita) berdasarkan frekuensi kata tersebut di seluruh dataset.


In [10]:
def model_tf_idf(data, model, kategori):
	tfidf_matrix = model.transform(data)
	feature_names = model.get_feature_names_out()

	df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
	df_tfidf.insert(0, 'Kategori Berita', kategori.reset_index(drop=True))

	return df_tfidf

df_tfidf_test = model_tf_idf(data_test['cleaned_text'], tfidf_model, data_test['Kategori'])

### Penjelasan Kode

1. **Fungsi `model_tf_idf`**:
   - **Parameter**:
     - `data`: Kumpulan teks yang telah dibersihkan (misalnya, ringkasan berita).
     - `model`: Model TF-IDF yang sudah dilatih sebelumnya (dalam hal ini, `tfidf_model`).
     - `kategori`: Kategori berita yang terkait dengan teks.
   - **Proses**:
     - `model.transform(data)`: Menggunakan model TF-IDF untuk menghitung matriks TF-IDF dari data baru (teks yang belum diolah).
     - `get_feature_names_out()`: Mengambil nama fitur (kata) dari model TF-IDF.
     - `pd.DataFrame(...)`: Mengonversi matriks TF-IDF menjadi DataFrame, dengan kolom fitur sebagai nama kata.
     - `insert(...)`: Menambahkan kolom baru yang berisi kategori berita ke DataFrame TF-IDF. Kolom ini disisipkan di awal.

2. **Menggunakan Fungsi**:
   - `df_tfidf_test = model_tf_idf(data_test['cleaned_text'], tfidf_model, data_test['Kategori'])`:
     - Memanggil fungsi `model_tf_idf` dengan data uji yang sudah dibersihkan (`data_test['cleaned_text']`), model TF-IDF yang telah dilatih (`tfidf_model`), dan kategori berita dari data uji (`data_test['Kategori']`).
     - Hasilnya adalah DataFrame TF-IDF (`df_tfidf_test`), yang berisi skor TF-IDF untuk setiap kata di setiap dokumen dalam data uji, beserta kategori berita yang bersangkutan.

### Tujuan
Tujuan dari kode ini adalah untuk:
- Menghitung matriks TF-IDF untuk data uji menggunakan model TF-IDF yang telah dilatih sebelumnya.
- Memastikan bahwa data baru (data uji) diolah dengan cara yang sama seperti data pelatihan, sehingga dapat digunakan untuk analisis lebih lanjut atau untuk klasifikasi.



In [11]:
df_tfidf_test.head()

,Kategori Berita,acung,adhi,administrasi,akademi,akademisi,akmil,aktif,aktivitas,aku,...,utup,wahyuningsih,wakil,warga,wbk,wib,widodo,wilayah,yayasan,zona
0,Politik,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.255991,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Peristiwa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,News,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.417029,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Peristiwa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.252205,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Peristiwa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0


```python
df_tfidf_test.head()
```

### Penjelasan
1. **Fungsi `head()`**:
   - Fungsi ini digunakan untuk menampilkan beberapa baris teratas dari DataFrame. Secara default, `head()` menampilkan 5 baris pertama, tetapi Anda dapat mengubah jumlah baris yang ditampilkan dengan memberikan argumen, seperti `head(10)` untuk menampilkan 10 baris.

2. **Output**:
   - Hasil dari perintah `df_tfidf_test.head()` akan menampilkan DataFrame yang berisi kolom-kolom seperti:
     - **Kategori Berita**: Kategori dari berita (misalnya, politik, olahraga, dll.).
     - **Istilah/Kata**: Nama-nama kata yang terdapat dalam teks yang diolah.
     - **Nilai TF-IDF**: Skor TF-IDF untuk setiap istilah dalam setiap dokumen.


In [12]:
df_tfidf

,Kategori Berita,acung,adhi,administrasi,akademi,akademisi,akmil,aktif,aktivitas,aku,...,utup,wahyuningsih,wakil,warga,wbk,wib,widodo,wilayah,yayasan,zona
0,Peristiwa,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.200927,0.000000
1,Politik,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,Peristiwa,0.000000,0.000000,0.251956,0.000000,0.000000,0.000000,0.000000,0.251956,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,Peristiwa,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,Peristiwa,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.203137,0.000000,0.000000,0.000000,0.000000,0.203137,0.000000,0.000000,0.000000,0.000000
5,Politik,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,Peristiwa,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,Megapolitan,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,Peristiwa,0.000000,0.326772,0.000000,0.326772,0.000000,0.326772,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,News,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


DataFrame df_tfidf berisi hasil transformasi TF-IDF dari data latih (training data) yang telah diproses. DataFrame ini terdiri dari kolom-kolom yang menunjukkan kategori berita dan skor TF-IDF untuk setiap istilah yang terdapat dalam teks berita.

In [13]:
df_tfidf.to_csv("data_training_vsm.csv", index=False)
df_tfidf_test.to_csv("data_testing_vsm.csv", index=False)

- File data_training_vsm.csv:
Akan berisi data TF-IDF dari data pelatihan (df_tfidf), dengan kolom kategori dan skor TF-IDF untuk setiap istilah.
- File data_testing_vsm.csv:
Akan berisi data TF-IDF dari data pengujian (df_tfidf_test), dengan format yang sama seperti data pelatihan.

In [14]:
with open('tfidf_model.pkl', 'wb') as f:
    pickle.dump(tfidf_model, f)

### Penjelasan Kode
```python
with open('tfidf_model.pkl', 'wb') as f:
    pickle.dump(tfidf_model, f)
```

1. **`with open('tfidf_model.pkl', 'wb') as f:`**:
   - Ini membuka file `tfidf_model.pkl` untuk ditulis (`'wb'` berarti "write binary"). Menggunakan konteks `with` memastikan bahwa file akan ditutup secara otomatis setelah blok kode selesai dijalankan, bahkan jika terjadi kesalahan.

2. **`pickle.dump(tfidf_model, f)`**:
   - Fungsi `pickle.dump()` digunakan untuk menyimpan objek Python ke dalam file. Dalam hal ini, objek `tfidf_model`, yang merupakan model TF-IDF yang telah dilatih, disimpan ke dalam file `f`.
